In [1]:
import duckdb
import pandas as pd
from datetime import date

In [2]:
con = duckdb.connect()

con.execute("""
    CREATE TABLE employees AS
    SELECT * FROM read_csv_auto('../data/clean/employees.csv')
""")

con.execute("""
    CREATE TABLE positions AS
    SELECT * FROM read_csv_auto('../data/clean/positions.csv')
""")

con.execute("""
    CREATE TABLE hr_tickets AS
    SELECT * FROM read_csv_auto('../data/clean/hr_tickets.csv')
""")

print("Loaded:")
print(f"  employees:  {con.execute('SELECT COUNT(*) FROM employees').fetchone()[0]} rows")
print(f"  positions:  {con.execute('SELECT COUNT(*) FROM positions').fetchone()[0]} rows")
print(f"  hr_tickets: {con.execute('SELECT COUNT(*) FROM hr_tickets').fetchone()[0]} rows")

Loaded:
  employees:  419 rows
  positions:  24 rows
  hr_tickets: 2728 rows


In [3]:
dashboard_data = con.execute("""
    SELECT
        e.employee_id,
        e.dept,
        e.salary,
        e.hire_date,
        e.manager_id,
        e.salary_outlier,
        p.band_min,
        p.band_max,
        p.level,
        (p.band_min + p.band_max) / 2.0                    AS band_midpoint,
        (e.salary - (p.band_min + p.band_max) / 2.0)
            / ((p.band_min + p.band_max) / 2.0)            AS gap_pct,
        CASE
            WHEN e.manager_id IS NULL THEN 0
            ELSE 1
        END                                                 AS has_manager,
        DATEDIFF('year', CAST(e.hire_date AS DATE), TODAY()) AS tenure_years
    FROM employees e
    JOIN positions p ON e.position_id = p.position_id
""").df()

print(f"Rows: {len(dashboard_data)}")
print(dashboard_data[['dept','salary','band_midpoint','gap_pct','has_manager','tenure_years']].head(8))

Rows: 419
          dept  salary  band_midpoint   gap_pct  has_manager  tenure_years
0        Sales  102500       113400.0 -0.096120            1             1
1  Engineering   96000       105300.0 -0.088319            1             1
2  Engineering  128000       125000.0  0.024000            1             1
3  Engineering  116000       125000.0 -0.072000            1             1
4         Data  186000       180000.0  0.033333            1             1
5      Product  171000       162500.0  0.052308            1             1
6  Engineering  160500       160000.0  0.003125            1             1
7         Data  152500       157500.0 -0.031746            1             1


In [4]:
def tenure_bucket(years):
    if years < 1:
        return "< 1 year"
    elif years < 3:
        return "1–3 years"
    elif years < 5:
        return "3–5 years"
    else:
        return "5+ years"

dashboard_data['tenure_bucket'] = dashboard_data['tenure_years'].apply(tenure_bucket)

dashboard_data.to_csv('../data/clean/dashboard_data.csv', index=False)

print(f"Exported dashboard_data.csv — {len(dashboard_data)} rows")
print(f"\nTenure distribution:")
print(dashboard_data['tenure_bucket'].value_counts())
print(f"\nAvg gap_pct by dept (the Panel 1 story):")
print(
    dashboard_data.groupby('dept')['gap_pct']
    .mean()
    .sort_values()
    .map(lambda x: f"{x:.1%}")
)

Exported dashboard_data.csv — 419 rows

Tenure distribution:
tenure_bucket
3–5 years    203
1–3 years    152
5+ years      64
Name: count, dtype: int64

Avg gap_pct by dept (the Panel 1 story):
dept
Customer Success    -14.8%
Marketing           -11.8%
Finance             -11.6%
HR                  -10.9%
Sales                -3.6%
Executive            -1.5%
Engineering          -1.1%
Product              -0.5%
Data                  0.2%
Name: gap_pct, dtype: object


In [5]:
tickets_analysis = con.execute("""
    SELECT
        t.ticket_id,
        t.category,
        t.status,
        t.open_date,
        t.close_date,
        e.dept,
        DATEDIFF('day',
            CAST(t.open_date  AS DATE),
            CAST(t.close_date AS DATE)
        ) AS resolution_days
    FROM hr_tickets t
    LEFT JOIN employees e ON t.employee_id = e.employee_id
    WHERE t.status = 'Closed'
      AND t.close_date IS NOT NULL
      AND t.open_date  IS NOT NULL
""").df()

tickets_analysis.to_csv('../data/clean/tickets_analysis.csv', index=False)

print(f"Exported tickets_analysis.csv — {len(tickets_analysis)} rows")
print(f"\nMedian resolution_days by category:")
print(
    tickets_analysis.groupby('category')['resolution_days']
    .median()
    .sort_values(ascending=False)
    .map(lambda x: f"{x:.0f} days")
)

Exported tickets_analysis.csv — 2343 rows

Median resolution_days by category:
category
HR Technology             43 days
Employee Relations        34 days
Learning & Development    30 days
Compensation              21 days
Payroll                   19 days
Onboarding                16 days
Offboarding               12 days
Policy & Benefits         10 days
Name: resolution_days, dtype: object
